In [1]:
# imports

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import settings
from src.api import BOEDownloader
from src.preprocess import Preprocesador, generar_esquemas

from src.llm import Llm

from dotenv import load_dotenv

load_dotenv()

print("✓ Imports OK")
print(f"  Neo4j URI:  {settings.neo4j.uri}")
print(f"  LLM model:  {settings.llm.model}")
print(f"  Max exchanges: {settings.llm.max_exchanges}")

✓ Imports OK
  Neo4j URI:  bolt://localhost:7687
  LLM model:  claude-haiku-4-5
  Max exchanges: 1


In [2]:
# API

api = BOEDownloader()
resumen = api.descargar_masivo()
resumen

2026-06-01 21:05:33 [info     ] 
Obteniendo ids...            
2026-06-01 21:05:57 [info     ] Listado pagina                 obtenidos=10000 offset=0
2026-06-01 21:06:00 [info     ] Listado pagina                 obtenidos=2286 offset=10000
2026-06-01 21:06:00 [info     ] 
Ids persistidos               path=/home/jorgee/reversa/ontology/kinetic-layer/api_boe/ids.txt total=12286
2026-06-01 21:06:00 [info     ] 
Procesando ids...            


100%|██████████| 12286/12286 [15:02<00:00, 13.62norma/s, BOE-A-2026-11627] 

2026-06-01 21:21:02 [info     ] 
Descarga masiva completada    descargados=12286 fallidos=0 saltados=0 total=12286
2026-06-01 21:21:02 [info     ] 
Reintento completado          recuperados=0 total=0


ResumenDescarga(total=12286, descargados=12286, fallidos=0, saltados=0)

In [2]:
# Preprocesar — parsear XMLs y cargar en Neo4j

preprocesador = Preprocesador()
resumen = preprocesador.preprocesar_todo()
resumen

2026-06-02 00:46:20 [info     ] Grafo limpiado                
2026-06-02 00:46:20 [info     ] 
Preprocesando...             


100%|██████████| 12286/12286 [11:18<00:00, 18.12norma/s, BOE-A-2026-9961]  

2026-06-02 00:57:39 [info     ] 
Reintento completado          recuperados=0 total=0


2026-06-02 00:57:39 [info     ] Faltantes escritos             anteriores=10082 posteriores=7619
2026-06-02 00:57:39 [info     ] 
Esquemas creados              semantic_dir=/home/jorgee/reversa/ontology/semantic-layer
2026-06-02 00:57:39 [info     ] 
Preprocesado completado       aristas=52086 errores=0 nodos=12286 procesadas=12286


ResumenPreproc(procesadas=12286, nodos_upsert=12286, aristas_upsert=52086, errores=0)

In [4]:
def borrar_nodos_dinamicos_bbdd() -> dict:
    """Elimina todos los nodos :UserQuery y aristas :RESULT_EDGE del grafo.

    Útil para limpiar el grafo dinámico entre sesiones de prueba.

    Returns:
        Resumen con el número de nodos y aristas eliminados.
    """
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        settings.neo4j.uri,
        auth=(settings.neo4j.user, settings.neo4j.password),
    )
    with driver.session(database=settings.neo4j.database) as session:
        result = session.run("MATCH (q:UserQuery) DETACH DELETE q RETURN count(q) AS eliminados")
        eliminados = result.single()["eliminados"]
    driver.close()
    return {"nodos_eliminados": eliminados}


print(borrar_nodos_dinamicos_bbdd())

{'nodos_eliminados': 9}


In [3]:
# Neo4j — stats rápidas del grafo
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    settings.neo4j.uri,
    auth=(settings.neo4j.user, settings.neo4j.password),
)
with driver.session(database=settings.neo4j.database) as session:
    def q(cypher):
        return session.run(cypher).single()["c"]

    stats = {
        "total_normas":               q("MATCH (n:Norma) RETURN count(n) AS c"),
        "normas_consolidadas":        q("MATCH (n:Norma) WHERE size(keys(n)) > 1 RETURN count(n) AS c"),
        "no_consolidadas_origen":     q("MATCH (a:Norma)-[]->(b:Norma) WHERE size(keys(a)) = 1 AND size(keys(b)) > 1 RETURN count(DISTINCT a) AS c"),
        "no_consolidadas_destino":    q("MATCH (a:Norma)-[]->(b:Norma) WHERE size(keys(a)) > 1 AND size(keys(b)) = 1 RETURN count(DISTINCT b) AS c"),
        "vigentes":                   q("MATCH (n:Norma {vigente: true}) RETURN count(n) AS c"),
        "total_aristas":              q("MATCH ()-[r]->() RETURN count(r) AS c"),
        "aristas_CITA":               q("MATCH ()-[:CITA]->() RETURN count(*) AS c"),
        "aristas_DEROGA":             q("MATCH ()-[:DEROGA]->() RETURN count(*) AS c"),
        "aristas_MODIFICA":           q("MATCH ()-[:MODIFICA]->() RETURN count(*) AS c"),
        "user_queries":               q("MATCH (q:UserQuery) RETURN count(q) AS c"),
    }
driver.close()

print("Stats del grafo:")
for k, v in stats.items():
    print(f"  {k:<30} {v:>8,}")

Stats del grafo:
  total_normas                     29,012
  normas_consolidadas              12,286
  no_consolidadas_origen            7,619
  no_consolidadas_destino          10,082
  vigentes                          9,910
  total_aristas                    52,086
  aristas_CITA                     14,443
  aristas_DEROGA                   15,944
  aristas_MODIFICA                 21,699
  user_queries                          0


In [4]:
# Generar esquemas semánticos

generar_esquemas()

2026-06-01 21:02:55 [info     ] 
Esquemas creados             


In [11]:
# LLM — prueba de una sola pregunta
async def test_llm_simple(pregunta:str = "¿Cuántas normas vigentes hay en el grafo?"):
    llm = Llm()
    print(f"Pregunta: {pregunta}\n")
    print("Respuesta: ", end="", flush=True)
    async for token in llm.responder(pregunta):
        print(token, end="", flush=True)
    print()
    llm.close()


await test_llm_simple()

Pregunta: ¿Cuántas normas vigentes hay en el grafo?

Respuesta: 

El grafo contiene **9.910 normas vigentes** actualmente.


In [7]:
# LLM — prueba de conversación multi-turno (max_exchanges)
async def test_llm_conversacion():
    llm = Llm()
    preguntas = [
        "¿Cuántas leyes vigentes hay en el grafo?",
        "¿Y cuántos reales decretos?",
        "¿Cuál es la norma más citada?",  # este turno descarta el primero del historial
    ]
    for i, pregunta in enumerate(preguntas, 1):
        print(f"--- Turno {i}: {pregunta}")
        print("Respuesta: ", end="", flush=True)
        async for token in llm.responder(pregunta):
            print(token, end="", flush=True)
        print(f"\n[historial: {len(llm._history)} exchange(s)]\n")
    llm.close()


await test_llm_conversacion()

--- Turno 1: ¿Cuántas leyes vigentes hay en el grafo?
Respuesta: El grafo contiene **3.397 leyes vigentes** actualmente.
[historial: 1 exchange(s)]

--- Turno 2: ¿Y cuántos reales decretos?
Respuesta: El grafo contiene **2.674 reales decretos vigentes** actualmente.
[historial: 2 exchange(s)]

--- Turno 3: ¿Cuál es la norma más citada?
Respuesta: La norma más citada en el grafo es:

**[BOE-A-1992-26318 — Ley 30/1992, de 26 de noviembre, de Régimen Jurídico de las Administraciones Públicas y del Procedimiento Administrativo Común]**

Esta norma es citada por **319 normas diferentes** en el grafo, lo que la convierte en la más referenciada del sistema.
[historial: 2 exchange(s)]



## Briefing

##### 1 Diagnosis: which laws have become unreadable? (The 5 norms most amended by other norms.)

In [10]:
pregunta = """
The Vice-Presidency wants the consolidation backlog: the laws amended so many times they are now incomprehensible even to lawyers. Give the top 5. These are the first candidates for a clean rewrite.
"""
await test_llm_simple(pregunta)

Pregunta: 
The Vice-Presidency wants the consolidation backlog: the laws amended so many times they are now incomprehensible even to lawyers. Give the top 5. These are the first candidates for a clean rewrite.


Respuesta: ## Consolidation Backlog: Top 5 Laws Requiring Rewrite

Here are the five vigente laws with the highest number of modifications—prime candidates for comprehensive legislative rewriting:

| Rank | Norma | Modificaciones | Fecha Original |
|------|-------|-----------------|-----------------|
| 1 | [BOE-A-2003-23936 — Ley 62/2003, de 30 de diciembre, de medidas fiscales, administrativas y del orden social] | **88** | 2003-12-30 |
| 2 | [BOE-A-2002-25412 — Ley 53/2002, de 30 de diciembre, de Medidas Fiscales, Administrativas y del Orden Social] | **74** | 2002-12-30 |
| 3 | [BOE-A-2025-1 — Ley 6/2024, de 5 de diciembre, de simplificación administrativa] | **68** | 2024-12-05 |
| 4 | [BOE-A-2000-24357 — Ley 14/2000, de 29 de diciembre, de Medidas fiscales, administrativas

##### 2 Root cause: who made the mess? (The 5 norms that amend the most other norms.)

In [11]:
pregunta = """
Show the Council how laws become unreadable. Find the “omnibus” laws — single acts that silently rewrote dozens of unrelated statutes at once. Name the top 5 worst offenders. The Council wants to see the pattern, not just the symptom.
"""
await test_llm_simple(pregunta)

Pregunta: 
Show the Council how laws become unreadable. Find the “omnibus” laws — single acts that silently rewrote dozens of unrelated statutes at once. Name the top 5 worst offenders. The Council wants to see the pattern, not just the symptom.


Respuesta: ---

## **THE OMNIBUS PATTERN: TOP 5 WORST OFFENDERS**

### **The Council's Evidence**

The graph reveals a systematic legislative pattern that obscures legal change: **five "year-end fiscal laws" that silently rewrote dozens of unrelated statutes in single acts**. This is the omnibus problem in its purest form.

---

### **TOP 5 OMNIBUS LAWS**

| Rank | Norma | Modificaciones | Patrón |
|------|-------|-----------------|--------|
| **1** | [BOE-A-2003-23936 — Ley 62/2003, de 30 de diciembre, de medidas fiscales, administrativas y del orden social] | **88 normas** | 49 Leyes + 20 normas sin rango + 11 RDL |
| **2** | [BOE-A-2002-25412 — Ley 53/2002, de 30 de diciembre, de Medidas Fiscales, Administrativas y del Orden Social] | **74

##### 3 The rot: how much of the statute book rests on dead law? (The percentage of in-force norms that cite at least one repealed norm, and the 5 repealed norms most cited by in-force norms.)

In [12]:
pregunta = """
Find every law still in force that cites a law already repealed. Quantify it: what fraction of live Spanish law rests on legal ground that no longer exists? Then surface the top 5 most-cited ghosts — the dead laws still propping up the most live statutes.
"""
await test_llm_simple(pregunta)

Pregunta: 
Find every law still in force that cites a law already repealed. Quantify it: what fraction of live Spanish law rests on legal ground that no longer exists? Then surface the top 5 most-cited ghosts — the dead laws still propping up the most live statutes.


Respuesta: Perfect. Now let me calculate the fraction and present the findings:

---

## **Spanish Law Built on Ghosts: A Structural Vulnerability**

### **The Numbers**

- **Total repealed/annulled laws**: 1,948
- **Total live laws**: 9,908
- **Live laws citing dead laws**: 772
- **Fraction of live law resting on repealed ground**: **7.8%** (772 out of 9,908)

This means **nearly 1 in 13 Spanish laws still in force explicitly cite legal provisions that no longer exist**. This represents a significant structural coherence problem in the Spanish legal system.

---

### **The Top 5 "Ghost Laws" — Dead Statutes Still Propping Up Live Ones**

| Rank | Repealed Law | Citations from Live Laws | Impact |
|------|---|---|---|
| 1

##### 4 The scalpel: the unfinished repeal. (The list of in-force norms that cite Ley 30/1992 directly.)

In [13]:
pregunta = """
In 2015 the Council repealed Ley 30/1992, the act on the Legal Regime of Public Administrations and Common Administrative Procedure, and replaced it with Leyes 39/2015 and 40/2015. But the cleanup was never finished: dozens of laws still in force keep citing Ley 30/1992 as if it existed. The Council wants to close the operation — update those orphan references — and needs the worklist. Compute the blast radius of Ley 30/1992: the laws still in force that cite it directly.
"""
await test_llm_simple(pregunta)

Pregunta: 
In 2015 the Council repealed Ley 30/1992, the act on the Legal Regime of Public Administrations and Common Administrative Procedure, and replaced it with Leyes 39/2015 and 40/2015. But the cleanup was never finished: dozens of laws still in force keep citing Ley 30/1992 as if it existed. The Council wants to close the operation — update those orphan references — and needs the worklist. Compute the blast radius of Ley 30/1992: the laws still in force that cite it directly.


Respuesta: Excellent! I've found the "blast radius" of Ley 30/1992. Here's the worklist for the Council:

## **Orphan References to Ley 30/1992 — Vigent Laws Requiring Update**

**Total count: 400 laws still in force that cite Ley 30/1992 directly**

### Key Findings:

The graph shows that despite the 2015 repeal of [BOE-A-1992-XXXXX — Ley 30/1992], a substantial number of laws across all administrative levels continue to reference it. These span:

- **Leyes** (Laws) — including state-level legislation
- 